# 107 --- Full Trade Stream: Real-Time Trade Analysis

Subscribe to the full trade stream for stocks or options and analyze trade
flow in real time. See unusual activity, large blocks, momentum shifts.

**Architecture:**
- `ThetaDataDx` streams decoded trade/quote events at wire speed
- FIT decoding and delta decompression happen in Rust -- Python receives named fields
- No raw payloads, no manual decoding -- `event["price"]`, `event["size"]`, etc.

**Note:** Full trade streaming requires a ThetaData subscription with real-time
access. Data is only available during market hours (9:30 AM -- 4:00 PM ET).

```
pip install thetadatadx[pandas] matplotlib
```

In [ ]:
import time
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from thetadatadx import (
    Credentials, Config, ThetaDataDx, to_dataframe,
)

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

# --- Authenticate and connect ---
creds = Credentials("your-email@example.com", "your-password")
config = Config.production()

tdx = ThetaDataDx(creds, config)
tdx.start_streaming()
print("ThetaDataDx:", tdx)

login_ev = tdx.next_event(timeout_ms=5000)
if login_ev and login_ev["kind"] == "login_success":
    print(f"FPSS authenticated! Permissions: {login_ev['detail']}")
else:
    print(f"Unexpected first event: {login_ev}")

---
## Part 2: Subscribe and Collect Decoded Trades

Subscribe to trade streams for a watchlist of symbols. The Rust core decodes
FIT payloads and delta-decompresses ticks before they reach Python. Each event
dict has named fields: `price`, `size`, `exchange`, `condition`, `ms_of_day`, etc.

In [ ]:
# --- Subscribe to trade streams for a watchlist ---

WATCHLIST = ["AAPL", "MSFT", "NVDA", "TSLA", "SPY", "QQQ", "AMD", "AMZN", "META", "GOOG"]

# Build contract_id -> symbol mapping from contract_assigned events
contracts = {}

sub_ids = {}
for sym in WATCHLIST:
    req_id = tdx.subscribe_trades(sym)
    sub_ids[sym] = req_id
    print(f"  Subscribed {sym} -> req_id={req_id}")

# Drain confirmation + contract assignment events
confirmations = 0
while True:
    ev = tdx.next_event(timeout_ms=2000)
    if ev is None:
        break
    if ev["kind"] == "req_response":
        confirmations += 1
    elif ev["kind"] == "contract_assigned":
        contracts[ev["id"]] = ev["detail"]
    elif ev["kind"] in ("error", "server_error"):
        print(f"  Warning: {ev['kind']}: {ev.get('detail')}")

print(f"\nConfirmed {confirmations}/{len(WATCHLIST)} subscriptions")
print(f"Contract map: {contracts}")

In [ ]:
# --- Collect decoded trade events for 30 seconds ---
#
# Events arrive with all fields pre-decoded by the Rust core:
#   event["kind"] == "trade"
#   event["contract_id"], event["price"], event["size"],
#   event["ms_of_day"], event["exchange"], event["condition"], etc.

COLLECT_SECONDS = 30

trades = []
other_events = defaultdict(int)
deadline = time.monotonic() + COLLECT_SECONDS

print(f"Collecting decoded trade events for {COLLECT_SECONDS}s across {len(WATCHLIST)} symbols...")
while time.monotonic() < deadline:
    remaining_ms = int((deadline - time.monotonic()) * 1000)
    if remaining_ms <= 0:
        break
    ev = tdx.next_event(timeout_ms=min(remaining_ms, 500))
    if ev is None:
        continue
    if ev["kind"] == "trade":
        trades.append({
            "symbol": contracts.get(ev["contract_id"], "???"),
            "contract_id": ev["contract_id"],
            "price": ev["price"],
            "size": ev["size"],
            "exchange": ev["exchange"],
            "condition": ev["condition"],
            "ms_of_day": ev["ms_of_day"],
            "sequence": ev["sequence"],
            "date": ev["date"],
        })
    elif ev["kind"] == "contract_assigned":
        contracts[ev["id"]] = ev["detail"]
    else:
        other_events[ev["kind"]] += 1

print(f"\nCollected {len(trades):,} decoded trades in {COLLECT_SECONDS}s")
print(f"Throughput: {len(trades) / COLLECT_SECONDS:.1f} trades/sec")
if other_events:
    print(f"Other events: {dict(other_events)}")

# Build DataFrame directly from decoded stream events
df_trades = pd.DataFrame(trades)

if not df_trades.empty:
    df_trades["time"] = pd.to_timedelta(df_trades["ms_of_day"], unit="ms")
    df_trades["time_str"] = df_trades["time"].apply(
        lambda td: f"{int(td.total_seconds()//3600):02d}:"
                   f"{int((td.total_seconds()%3600)//60):02d}:"
                   f"{td.total_seconds()%60:06.3f}"
    )
    df_trades["notional"] = df_trades["price"] * df_trades["size"]
    print(f"\nColumns: {list(df_trades.columns)}")
    print()
    display(df_trades[["time_str", "symbol", "price", "size", "exchange", "condition"]].tail(20))
else:
    print("No trades received (market may be closed).")

---
## Part 3: Real-Time Trade Analysis

Four views: volume by symbol, large block detection, price momentum, and buy/sell imbalance.

In [ ]:
# --- Volume by symbol (bar chart) + Large block detector ---

BLOCK_THRESHOLD = 1000  # shares

if not df_trades.empty:
    # Volume by symbol
    vol_by_sym = (
        df_trades.groupby("symbol")["size"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    vol_by_sym.plot(kind="bar", ax=ax, color="#1f77b4", edgecolor="white")
    ax.set_title("Total Volume by Symbol (Today)")
    ax.set_ylabel("Shares")
    ax.set_xlabel("")
    ax.bar_label(ax.containers[0], fmt="{:,.0f}", fontsize=9)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Large block detector
    blocks = df_trades[df_trades["size"] > BLOCK_THRESHOLD].copy()
    blocks["notional"] = blocks["price"] * blocks["size"]

    print(f"\nLarge blocks (>{BLOCK_THRESHOLD:,} shares): {len(blocks):,} trades")
    if not blocks.empty:
        print(f"Total block volume: {blocks['size'].sum():,.0f} shares")
        print(f"Total block notional: ${blocks['notional'].sum():,.2f}")
        print()
        display(
            blocks.nlargest(15, "notional")[
                ["time_str", "symbol", "price", "size", "notional", "exchange", "condition"]
            ].reset_index(drop=True)
        )
    else:
        print("No large blocks detected.")
else:
    print("No trade data available.")

In [ ]:
# --- Price momentum + Buy/sell imbalance for the most active symbol ---

if not df_trades.empty:
    most_active = df_trades.groupby("symbol")["size"].sum().idxmax()
    sym_trades = df_trades[df_trades["symbol"] == most_active].sort_values("ms_of_day").copy()

    # --- Price momentum (RTH only) ---
    RTH_START = 9 * 3600000 + 30 * 60000  # 9:30 AM in ms
    RTH_END = 16 * 3600000               # 4:00 PM in ms
    rth = sym_trades[
        (sym_trades["ms_of_day"] >= RTH_START) & (sym_trades["ms_of_day"] <= RTH_END)
    ]

    if not rth.empty:
        rth = rth.copy()
        rth["second"] = rth["ms_of_day"] // 1000
        ohlc = rth.groupby("second")["price"].agg(["first", "max", "min", "last"])

        fig, ax = plt.subplots(figsize=(14, 5))
        hours = ohlc.index / 3600
        ax.plot(hours, ohlc["last"], linewidth=0.7, color="#1f77b4", alpha=0.9)
        ax.fill_between(hours, ohlc["min"], ohlc["max"], alpha=0.15, color="#1f77b4")
        ax.set_title(f"{most_active} Intraday Price from Trade Tape")
        ax.set_xlabel("Hour (ET)")
        ax.set_ylabel("Price ($)")
        ax.set_xlim(9.5, 16.0)
        plt.tight_layout()
        plt.show()
    else:
        print(f"No RTH trades for {most_active}.")

    # --- Buy/sell imbalance (tick rule) ---
    #
    # Uptick -> buy aggressor, downtick -> sell aggressor,
    # zero-tick -> carry forward previous classification.
    side = []
    prev_price = None
    prev_side = "buy"
    for _, row in sym_trades.iterrows():
        p = row["price"]
        if prev_price is not None:
            if p > prev_price:
                prev_side = "buy"
            elif p < prev_price:
                prev_side = "sell"
        side.append(prev_side)
        prev_price = p

    sym_trades["side"] = side
    buy_vol = sym_trades[sym_trades["side"] == "buy"]["size"].sum()
    sell_vol = sym_trades[sym_trades["side"] == "sell"]["size"].sum()
    total = buy_vol + sell_vol

    print(f"\n{most_active} Buy/Sell Imbalance (tick rule):")
    print(f"  Buy volume:  {buy_vol:>12,} ({buy_vol/total*100:.1f}%)")
    print(f"  Sell volume: {sell_vol:>12,} ({sell_vol/total*100:.1f}%)")
    print(f"  Net:         {buy_vol - sell_vol:>+12,} shares")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Buy", "Sell"], [buy_vol, sell_vol],
           color=["#2ecc71", "#e74c3c"], edgecolor="white")
    ax.set_title(f"{most_active} Volume by Aggressor Side")
    ax.set_ylabel("Shares")
    ax.bar_label(ax.containers[0], fmt="{:,.0f}", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("No trade data available.")

---
## Part 4: Subscribe to Full Option Trades

Option trade streaming works per-contract: you specify underlying, expiration,
call/put, and strike. We pick the nearest expiration and a spread of strikes
around ATM to capture the most active contracts.

In [ ]:
# --- Subscribe to option trade streams for SPY near ATM ---

OPT_UNDERLYING = "SPY"

# Get spot price and nearest expiration
spot_snap = tdx.stock_snapshot_ohlc([OPT_UNDERLYING])
spot = spot_snap[0]["close"] if spot_snap else 0.0
print(f"{OPT_UNDERLYING} spot: ${spot:.2f}")

expirations = tdx.option_list_expirations(OPT_UNDERLYING)
nearest_exp = expirations[0]
exp_int = int(nearest_exp)
exp_dt = datetime.strptime(nearest_exp, "%Y%m%d")
dte = (exp_dt - datetime.now()).days
print(f"Nearest expiration: {nearest_exp} ({dte} DTE)")

# Get strikes near ATM
all_strikes = tdx.option_list_strikes(OPT_UNDERLYING, nearest_exp)
strike_vals = [(s, float(s) / 1000) for s in all_strikes]
atm_idx = min(range(len(strike_vals)), key=lambda i: abs(strike_vals[i][1] - spot))

# Subscribe to 5 strikes around ATM, both calls and puts
STRIKE_RANGE = 5
opt_sub_ids = []
start = max(0, atm_idx - STRIKE_RANGE)
end = min(len(strike_vals), atm_idx + STRIKE_RANGE + 1)

for strike_str, strike_val in strike_vals[start:end]:
    for is_call in [True, False]:
        right_str = "C" if is_call else "P"
        req_id = tdx.subscribe_option_trades(
            OPT_UNDERLYING, exp_int, is_call, int(strike_str)
        )
        opt_sub_ids.append((strike_str, is_call, req_id))
        print(f"  {OPT_UNDERLYING} {nearest_exp} {strike_val:.0f}{right_str} -> req_id={req_id}")

# Drain confirmations
opt_confirmations = 0
while True:
    ev = tdx.next_event(timeout_ms=2000)
    if ev is None:
        break
    if ev["kind"] == "req_response":
        opt_confirmations += 1

print(f"\nConfirmed {opt_confirmations}/{len(opt_sub_ids)} option subscriptions")

In [ ]:
# --- Collect decoded option + stock trade events for 30 seconds ---

OPT_COLLECT_SECONDS = 30

opt_trades = []
deadline = time.monotonic() + OPT_COLLECT_SECONDS

print(f"Collecting decoded trade events for {OPT_COLLECT_SECONDS}s (stocks + options)...")
while time.monotonic() < deadline:
    remaining_ms = int((deadline - time.monotonic()) * 1000)
    if remaining_ms <= 0:
        break
    ev = tdx.next_event(timeout_ms=min(remaining_ms, 500))
    if ev is None:
        continue
    if ev["kind"] == "trade":
        opt_trades.append({
            "contract_id": ev["contract_id"],
            "price": ev["price"],
            "size": ev["size"],
            "exchange": ev["exchange"],
            "condition": ev["condition"],
            "ms_of_day": ev["ms_of_day"],
        })
    elif ev["kind"] == "contract_assigned":
        contracts[ev["id"]] = ev["detail"]

total = len(opt_trades)
print(f"\nTotal decoded trade events: {total:,}")
print(f"Throughput: {total / OPT_COLLECT_SECONDS:.1f} trades/sec")

In [ ]:
# --- Pull decoded option trades + unusual activity + P/C ratio + premium flow ---
#
# option_snapshot_trade() returns decoded TradeTick dicts for each contract.

opt_trades = []
for strike_str, is_call, _ in opt_sub_ids:
    right = "C" if is_call else "P"
    strike_val = float(strike_str) / 1000
    try:
        snap = tdx.option_snapshot_trade(
            OPT_UNDERLYING, nearest_exp, strike_str, right
        )
        if snap:
            t = snap[0]
            t["underlying"] = OPT_UNDERLYING
            t["expiration"] = nearest_exp
            t["strike"] = strike_val
            t["right"] = right
            opt_trades.append(t)
    except Exception:
        pass

df_opt = pd.DataFrame(opt_trades)

if not df_opt.empty:
    df_opt["notional"] = df_opt["price"] * df_opt["size"] * 100  # options = 100 shares
    print(f"Option trade snapshots: {len(df_opt)} contracts with activity")
    display(
        df_opt[
            ["underlying", "expiration", "strike", "right", "price", "size", "notional"]
        ].sort_values("notional", ascending=False).reset_index(drop=True)
    )

    # --- Unusual activity detector ---
    UNUSUAL_THRESHOLD = 50_000  # $50k notional
    unusual = df_opt[df_opt["notional"] > UNUSUAL_THRESHOLD]
    print(f"\n=== UNUSUAL ACTIVITY (notional > ${UNUSUAL_THRESHOLD:,}) ===")
    print(f"Flagged contracts: {len(unusual)}")
    if not unusual.empty:
        display(
            unusual[
                ["underlying", "strike", "right", "price", "size", "notional"]
            ].sort_values("notional", ascending=False).reset_index(drop=True)
        )

    # --- Put/Call ratio ---
    calls = df_opt[df_opt["right"] == "C"]
    puts = df_opt[df_opt["right"] == "P"]
    call_vol = calls["size"].sum() if not calls.empty else 0
    put_vol = puts["size"].sum() if not puts.empty else 0
    pc_ratio = put_vol / call_vol if call_vol > 0 else float("inf")

    print(f"\n=== PUT/CALL RATIO ===")
    print(f"  Call volume: {call_vol:>8,} contracts")
    print(f"  Put volume:  {put_vol:>8,} contracts")
    print(f"  P/C ratio:   {pc_ratio:.3f}")

    # --- Premium flow ---
    call_premium = calls["notional"].sum() if not calls.empty else 0
    put_premium = puts["notional"].sum() if not puts.empty else 0
    net = call_premium - put_premium

    print(f"\n=== PREMIUM FLOW ===")
    print(f"  Call premium: ${call_premium:>14,.2f}")
    print(f"  Put premium:  ${put_premium:>14,.2f}")
    print(f"  Net (C - P):  ${net:>+14,.2f}")
    direction = "BULLISH (net call buying)" if net > 0 else "BEARISH (net put buying)"
    print(f"  Signal:       {direction}")

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(["Calls", "Puts"], [call_vol, put_vol],
               color=["#2ecc71", "#e74c3c"], edgecolor="white")
    axes[0].set_title(f"{OPT_UNDERLYING} Put/Call Volume (P/C = {pc_ratio:.2f})")
    axes[0].set_ylabel("Contracts")
    axes[0].bar_label(axes[0].containers[0], fmt="{:,.0f}")

    axes[1].bar(["Call $", "Put $"], [call_premium, put_premium],
               color=["#2ecc71", "#e74c3c"], edgecolor="white")
    axes[1].set_title(f"{OPT_UNDERLYING} Premium Flow (Net ${net:+,.0f})")
    axes[1].set_ylabel("Premium ($)")
    axes[1].bar_label(axes[1].containers[0], fmt="${:,.0f}")

    plt.tight_layout()
    plt.show()
else:
    print("No option trade snapshots returned (market may be closed).")

---
## Part 5: Combined Dashboard

Run a live-updating summary that polls both stock and option streams,
refreshing every 5 seconds. Uses `ipywidgets.ToggleButton` to stop
cleanly, or interrupt the kernel.

In [ ]:
# --- Live combined dashboard ---
#
# Collects decoded trade events in 5-second windows, pulls snapshots for context,
# and displays a rolling summary. Toggle the button or interrupt kernel to stop.

dashboard_output = widgets.Output()
stop_btn = widgets.ToggleButton(
    value=False, description="Stop Dashboard",
    button_style="danger", icon="stop",
)

display(stop_btn, dashboard_output)

REFRESH_INTERVAL = 5  # seconds
cumulative_trades = 0
largest_trade = {"symbol": "-", "size": 0, "price": 0.0, "notional": 0.0}

while not stop_btn.value:
    window_start = time.monotonic()
    window_trades = []

    # Collect decoded trade events for one window
    while time.monotonic() - window_start < REFRESH_INTERVAL:
        if stop_btn.value:
            break
        remaining_ms = int((REFRESH_INTERVAL - (time.monotonic() - window_start)) * 1000)
        if remaining_ms <= 0:
            break
        ev = tdx.next_event(timeout_ms=min(remaining_ms, 500))
        if ev is None:
            continue
        if ev["kind"] == "trade":
            window_trades.append(ev)
            notional = ev["price"] * ev["size"]
            sym = contracts.get(ev["contract_id"], "???")
            if notional > largest_trade["notional"]:
                largest_trade = {
                    "symbol": sym,
                    "size": ev["size"],
                    "price": ev["price"],
                    "notional": notional,
                }
        elif ev["kind"] == "contract_assigned":
            contracts[ev["id"]] = ev["detail"]

    cumulative_trades += len(window_trades)

    # Pull snapshot data for the dashboard
    stock_snaps = []
    for sym in WATCHLIST[:5]:
        try:
            snap = tdx.stock_snapshot_trade([sym])
            if snap:
                t = snap[0]
                t["symbol"] = sym
                stock_snaps.append(t)
        except Exception:
            pass

    opt_snaps = []
    for strike_str, is_call, _ in opt_sub_ids[:10]:
        right = "C" if is_call else "P"
        try:
            snap = tdx.option_snapshot_trade(
                OPT_UNDERLYING, nearest_exp, strike_str, right
            )
            if snap:
                t = snap[0]
                t["contract"] = f"{OPT_UNDERLYING} {float(strike_str)/1000:.0f}{right}"
                t["notional"] = t["price"] * t["size"] * 100
                opt_snaps.append(t)
        except Exception:
            pass

    call_prem = sum(t["notional"] for t in opt_snaps if "C" in t["contract"])
    put_prem = sum(t["notional"] for t in opt_snaps if "P" in t["contract"])

    # Render
    with dashboard_output:
        clear_output(wait=True)
        ts = time.strftime("%H:%M:%S")

        display(HTML(f"""
        <div style='font-family: monospace; padding: 10px;'>
        <h3>Trade Stream Dashboard &mdash; {ts}</h3>
        <p>Window trades: {len(window_trades):,} | Cumulative: {cumulative_trades:,} |
           Rate: {len(window_trades) / REFRESH_INTERVAL:.0f} trades/s</p>

        <h4>Top 5 Most Traded Stocks (last trade)</h4>
        <table border='1' cellpadding='5' style='border-collapse: collapse;'>
        <tr><th>Symbol</th><th>Price</th><th>Size</th><th>Exchange</th></tr>
        {''.join(
            f"<tr><td>{t['symbol']}</td><td>${t['price']:.2f}</td>"
            f"<td>{t['size']:,}</td><td>{t['exchange']}</td></tr>"
            for t in sorted(stock_snaps, key=lambda x: x['size'], reverse=True)[:5]
        )}
        </table>

        <h4>Top 5 Most Traded Options</h4>
        <table border='1' cellpadding='5' style='border-collapse: collapse;'>
        <tr><th>Contract</th><th>Price</th><th>Size</th><th>Notional</th></tr>
        {''.join(
            f"<tr><td>{t['contract']}</td><td>${t['price']:.2f}</td>"
            f"<td>{t['size']:,}</td><td>${t['notional']:,.0f}</td></tr>"
            for t in sorted(opt_snaps, key=lambda x: x['notional'], reverse=True)[:5]
        )}
        </table>

        <h4>Net Premium Flow</h4>
        <p>Call premium: ${call_prem:,.0f} | Put premium: ${put_prem:,.0f} |
           Net: <b>${call_prem - put_prem:+,.0f}</b>
           {'(BULLISH)' if call_prem > put_prem else '(BEARISH)'}</p>

        <h4>Largest Single Trade (session)</h4>
        <p>{largest_trade['symbol']} &mdash;
           {largest_trade['size']:,} shares @ ${largest_trade['price']:.2f}
           = ${largest_trade['notional']:,.2f}</p>
        </div>
        """))

print("Dashboard stopped.")

In [ ]:
# --- Part 6: Export and Cleanup ---

# Save collected trades to CSV
if not df_trades.empty:
    csv_path = f"stock_trades_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    export_cols = ["time_str", "symbol", "price", "size", "exchange",
                   "condition", "sequence"]
    df_trades[export_cols].to_csv(csv_path, index=False)
    print(f"Stock trades saved: {csv_path} ({len(df_trades):,} rows)")

if "df_opt" in dir() and not df_opt.empty:
    opt_csv = f"option_trades_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_opt.to_csv(opt_csv, index=False)
    print(f"Option trades saved: {opt_csv} ({len(df_opt):,} rows)")

# Unsubscribe and shut down
for sym in WATCHLIST:
    try:
        tdx.unsubscribe_trades(sym)
    except Exception:
        pass
print(f"\nUnsubscribed {len(WATCHLIST)} stock streams")

# Drain remaining events
drained = 0
while True:
    ev = tdx.next_event(timeout_ms=500)
    if ev is None:
        break
    drained += 1
print(f"Drained {drained} remaining events")

tdx.stop_streaming()
print(tdx)
print("\nClean shutdown complete.")

---

**Previous:** [106 --- Live Option Chain](106_live_option_chain.ipynb)  
**Back to start:** [101 --- Getting Started](101_getting_started.ipynb)